# 08 - Figure 4

Thin caller: shared data-load cells are replaced with `nfip` loader calls.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from src import config
from src.plotting import *   # shared figure style defaults

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

# Reference Dicts

In [ ]:
# FIPS mapping
state_fips_dict = config.FIPS_TO_NAME

# Abbreviation mapping (if needed)
state_abbrev = config.FIPS_TO_ABBREV

In [ ]:
nat_MHI_2023 = 82_690

income_df = pd.DataFrame({
    "State": config._STATES_51,
    "Median_Income_2023": config.MEDIAN_INCOME_2023
})

# Parameters

In [ ]:
save = True
input_clustering = '_new' # '_new' or ''

test_case = 'block'
premium_type1 = '' # '_Full' '_Discount' or ''
premium_type2 = '' # '_Full'  '_Discount' or ''
premium_type3 = '_Discount' # '_Full'  '_Discount' or ''


base_case1 = '_base_case'
base_case2 = ''
base_case3 = '_base_case'

# Data Load

## Simulations

In [ ]:
# Simulation data 1
state_balance_df1 = pd.read_csv(f'Results/state_balance_{test_case}{input_clustering}{premium_type1}{base_case1}.csv')
final_balances_df1 = pd.read_csv(f'Results/final_balances_{test_case}{input_clustering}{premium_type1}{base_case1}.csv')
cluster_state_df1 = pd.read_csv(f'Results/cluster_state_{test_case}{input_clustering}{premium_type1}{base_case1}.csv')
balance_transition_df1 = pd.read_csv(f'Results/balance_transition_{test_case}{input_clustering}{premium_type1}{base_case1}.csv')

# Simulation data 2
state_balance_df2 = pd.read_csv(f'Results/state_balance_{test_case}{input_clustering}{premium_type2}{base_case2}.csv')
final_balances_df2 = pd.read_csv(f'Results/final_balances_{test_case}{input_clustering}{premium_type2}{base_case2}.csv')
cluster_state_df2 = pd.read_csv(f'Results/cluster_state_{test_case}{input_clustering}{premium_type2}{base_case2}.csv')
balance_transition_df2 = pd.read_csv(f'Results/balance_transition_{test_case}{input_clustering}{premium_type2}{base_case2}.csv')

# Simulation data 3
state_balance_df3 = pd.read_csv(f'Results/state_balance_{test_case}{input_clustering}{premium_type3}{base_case3}.csv')
final_balances_df3 = pd.read_csv(f'Results/final_balances_{test_case}{input_clustering}{premium_type3}{base_case3}.csv')
cluster_state_df3 = pd.read_csv(f'Results/cluster_state_{test_case}{input_clustering}{premium_type3}{base_case3}.csv')
balance_transition_df3 = pd.read_csv(f'Results/balance_transition_{test_case}{input_clustering}{premium_type3}{base_case3}.csv')

## Risk Policies

In [ ]:
# Load risk policies
risk_policies = pd.read_excel('../Local_Data/NFIP_Data/nfip_policy-information-by-state_20240531.xlsx', sheet_name='PIF')

## Premium and Afford Compare

In [ ]:
premium_1 = (
    state_balance_df1
    .groupby("STATEFP")["premium"]
    .mean()
    .reset_index(name="premium")
)

premium_2 = (
    state_balance_df2
    .groupby("STATEFP")["premium"]
    .mean()
    .reset_index(name="premium")
)

premium_3 = (
    state_balance_df3
    .groupby("STATEFP")["premium"]
    .mean()
    .reset_index(name="premium")
)

# Merge the two dataframes by STATEFP
premium_compare = premium_3.merge(
    premium_2,
    on="STATEFP",
    suffixes=("_3", "_2")
)

# Compute the difference in the premium column
premium_compare["RR2_increase"] = premium_compare["premium_3"] - premium_compare["premium_2"]

In [ ]:
premium_compare["STATEFP"] = (
    premium_compare["STATEFP"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(2)
)
premium_compare["State"] = premium_compare["STATEFP"].map(state_abbrev)

In [ ]:
risk_policies['County'] = risk_policies['County'].str.strip()
risk_policies['State'] = risk_policies['State'].str.strip()

aggregated_risk_policies = risk_policies.groupby(['State']).agg({
    'Policies in Force': 'sum'
}).reset_index()

# Reverse the mapping: State Name → FIPS
state_name_to_fips = {v: k for k, v in state_fips_dict.items()}

aggregated_risk_policies["STATEFP"] = aggregated_risk_policies["State"].map(state_name_to_fips)

premium_compare = premium_compare.merge(income_df, on="State", how="left")

In [ ]:
premium_compare = premium_compare.merge(aggregated_risk_policies, on="STATEFP", how="left").rename(columns={
    "State_x": "State_Abbr",
    "State_y": "State"
})

In [ ]:
premium_compare['RR2_afford'] = premium_compare['premium_3']/premium_compare['Median_Income_2023']/premium_compare['Policies in Force']*100

premium_compare['base_afford'] = premium_compare['premium_2']/premium_compare['Median_Income_2023']/premium_compare['Policies in Force']*100

## Extreme Loss vs Underpricing Risk

In [ ]:
# Group by STATEFP and calculate averages
def average_ratio(df, flag_col, flag_val, label):
    return (
        df[df[flag_col] == flag_val]
        .groupby("STATEFP")["contribution_ratio"]
        .mean()
        .reset_index()
        .rename(columns={"contribution_ratio": label})
    )

In [ ]:
# Create category flags
state_balance_df1["is_positive"] = state_balance_df1["contribution"] > 0
state_balance_df1["is_negative"] = state_balance_df1["contribution"] < 0
state_balance_df1["contribution_ratio"] = state_balance_df1["contribution"] / state_balance_df1["premium"]
state_balance_df1.replace([np.inf, -np.inf], np.nan, inplace=True)

avg_positive1 = average_ratio(state_balance_df1, "is_positive", True, "Avg_Pos")
avg_negative1 = average_ratio(state_balance_df1, "is_negative", True, "Avg_Neg")

state_balance_df2["is_positive"] = state_balance_df2["contribution"] > 0
state_balance_df2["is_negative"] = state_balance_df2["contribution"] < 0
state_balance_df2["contribution_ratio"] = state_balance_df2["contribution"] / state_balance_df2["premium"]
state_balance_df2.replace([np.inf, -np.inf], np.nan, inplace=True)

avg_positive2 = average_ratio(state_balance_df2, "is_positive", True, "Avg_Pos")
avg_negative2 = average_ratio(state_balance_df2, "is_negative", True, "Avg_Neg")

state_balance_df3["is_positive"] = state_balance_df3["contribution"] > 0
state_balance_df3["is_negative"] = state_balance_df3["contribution"] < 0
state_balance_df3["contribution_ratio"] = state_balance_df3["contribution"] / state_balance_df3["premium"]
state_balance_df3.replace([np.inf, -np.inf], np.nan, inplace=True)

avg_positive3 = average_ratio(state_balance_df3, "is_positive", True, "Avg_Pos")
avg_negative3 = average_ratio(state_balance_df3, "is_negative", True, "Avg_Neg")

In [ ]:
# Normalize key dtype (strings like "01")
premium_compare["STATEFP"] = premium_compare["STATEFP"].astype(str).str.zfill(2)

for df_name in ["avg_positive1","avg_negative1","avg_positive2","avg_negative2","avg_positive3","avg_negative3"]:
    df = locals()[df_name]
    if "STATEFP" not in df.columns:  # handle Series/index cases
        df = df.reset_index()
    df["STATEFP"] = df["STATEFP"].astype(str).str.zfill(2)
    locals()[df_name] = df  # put back


# Rename columns for clarity
avg_positive1 = avg_positive1.rename(columns={"Avg_Pos": "Avg_Pos_1"})
avg_negative1 = avg_negative1.rename(columns={"Avg_Neg": "Avg_Neg_1"})

avg_positive2 = avg_positive2.rename(columns={"Avg_Pos": "Avg_Pos_2"})
avg_negative2 = avg_negative2.rename(columns={"Avg_Neg": "Avg_Neg_2"})

avg_positive3 = avg_positive3.rename(columns={"Avg_Pos": "Avg_Pos_3"})
avg_negative3 = avg_negative3.rename(columns={"Avg_Neg": "Avg_Neg_3"})

# Merge them all onto premium_compare
premium_compare = (
    premium_compare
    .merge(avg_positive1[["STATEFP", "Avg_Pos_1"]], on="STATEFP", how="left")
    .merge(avg_negative1[["STATEFP", "Avg_Neg_1"]], on="STATEFP", how="left")
    .merge(avg_positive2[["STATEFP", "Avg_Pos_2"]], on="STATEFP", how="left")
    .merge(avg_negative2[["STATEFP", "Avg_Neg_2"]], on="STATEFP", how="left")
    .merge(avg_positive3[["STATEFP", "Avg_Pos_3"]], on="STATEFP", how="left")
    .merge(avg_negative3[["STATEFP", "Avg_Neg_3"]], on="STATEFP", how="left")
)


# Plotting

## Timeseries of State Pools

In [ ]:
# Selected state abbreviations
selected_states = ['LA', 'MS', 'AL', 'TX', 'NY', 'NJ']

In [ ]:
def prep_state_stats(df, selected_states, state_abbrev):
    """
    Normalize STATEFP, compute cumulative balances per simulation/state, map to abbreviations,
    filter selected states, and aggregate stats across simulations by (State, year).
    Returns a DataFrame with columns: State, year, min, max, mean, q25, q75
    """
    df = df.copy()

    # Normalize STATEFP to zero-padded 2-char strings
    df["STATEFP"] = (
        df["STATEFP"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(2)
    )

    # cumulative balance per simulation & state
    df["cumulative_balance"] = (
        df.groupby(["simulation", "STATEFP"])["contribution"]
          .cumsum()
    )

    # Add abbreviation and filter
    df["State"] = df["STATEFP"].map(state_abbrev)
    df = df[df["State"].isin(selected_states)].copy()

    # Aggregate across simulations: min, max, mean, IQR
    stats = (
        df.groupby(["State", "year"])["cumulative_balance"]
          .agg(
              min="min",
              max="max",
              mean="mean",
              q25=lambda x: x.quantile(0.25),
              q75=lambda x: x.quantile(0.75),
          )
          .reset_index()
    )

    # Sort for clean plotting
    stats = stats.sort_values(["State", "year"])
    return stats

## Cost for changes

In [ ]:
reinsurance_costs = 88.43/3 + 50.37/3 + 61.23/3 + 121.1

RR2_costs = premium_compare[premium_compare["State_Abbr"].isin(selected_states)].copy()

RR2_costs['rein_increase'] = reinsurance_costs*RR2_costs['Policies in Force']/np.sum(RR2_costs['Policies in Force'])

RR2_costs['RR2_increase'] = RR2_costs['RR2_increase']/1_000_000

RR2_costs['rein_afford'] = (RR2_costs['premium_2'] + RR2_costs['rein_increase']*1_000_000)/premium_compare['Median_Income_2023']/premium_compare['Policies in Force']*100

In [ ]:
# Scenario cost/affordability table used by the final figure
df = RR2_costs.copy()

# Units: make sure cost columns are in millions
for col in ["RR2_increase", "rein_increase"]:
    if df[col].abs().max() > 1e5:  # likely raw dollars, not millions
        df[col] = df[col] / 1e6

In [ ]:
sns.set_theme(style="ticks", font_scale=0.8)

scenario_dfs = {
    "Base Case (2024)": state_balance_df1,
    "Reinsured": state_balance_df2,
    "Risk Rating (RR2)": state_balance_df3,
}

scenario_colors = {
    "Base Case (2024)": "#274B51", #554348
    "Reinsured": "#40909A",
    "Risk Rating (RR2)": "#C4403A", 
}

stats_by_scenario = {
    scen_name: prep_state_stats(df, selected_states, state_abbrev)
    for scen_name, df in scenario_dfs.items()
}

# Compute aggregate summaries
agg_cost = {
    "Base": df["premium_2"].sum() / 1e9,
    "Reinsured": (df["premium_2"].sum() / 1e9) + df["rein_increase"].sum()/1000,
    "RR2": (df["premium_2"].sum() / 1e9) + df["RR2_increase"].sum()/1000,
}

agg_afford = {
    "Base": df["base_afford"].mean(),
    "Reinsured": df["rein_afford"].mean(),
    "RR2": df["RR2_afford"].mean(),
}

agg_chronic = {  # Avg_Pos (Chronic)
    "Base": df["Avg_Pos_1"].mean(),
    "Reinsured": df["Avg_Pos_2"].mean(),
    "RR2": df["Avg_Pos_3"].mean(),
}

agg_extreme = {  # Avg_Neg (Extreme)
    "Base": df["Avg_Neg_1"].mean(),
    "Reinsured": df["Avg_Neg_2"].mean(),
    "RR2": df["Avg_Neg_3"].mean(),
}

# Figure layout
n_states = len(selected_states)
n_cols = 3
n_rows = -(-n_states // n_cols)

fig = plt.figure(figsize=(7, 7))
gs = GridSpec(nrows=2, ncols=1, height_ratios=[2, 1.15], hspace=0.4)

# Top: Line plots by state
gs_top = GridSpecFromSubplotSpec(nrows=n_rows, ncols=n_cols, subplot_spec=gs[0], wspace=0.25, hspace=0.45)

for i, state in enumerate(selected_states):
    r, c = divmod(i, n_cols)
    ax = fig.add_subplot(gs_top[r, c])

    for scen_name, stats in stats_by_scenario.items():
        color = scenario_colors.get(scen_name, "C0")
        df_sub = stats[stats["State"] == state].sort_values("year")
        if df_sub.empty:
            continue
        years = df_sub["year"].values

        # IQR shading
        ax.fill_between(years, df_sub["q25"]/(1e9), df_sub["q75"]/(1e9), color=color, alpha=0.35)
        # Mean line
        ax.plot(years, df_sub["mean"]/(1e9), color=color, lw=2.0, label=scen_name)

    ax.axhline(0, color="black", linestyle="--", lw=1)
    ax.set_title(f"{state} State Pool Balance")
    ax.grid(alpha=0.2)
    if r == n_rows - 1:
        ax.set_xlabel("Year")
    if c == 0:
        ax.set_ylabel("($B)")

# Shared legend
handles = [plt.Line2D([], [], color=color, lw=2, label=name) for name, color in scenario_colors.items()]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, title="Scenarios")

# Bottom: Aggregate barplots
gs_bot = GridSpecFromSubplotSpec(nrows=1, ncols=3, subplot_spec=gs[1], wspace=0.4, hspace = 0.5)

# Aggregate Cost
ax_cost = fig.add_subplot(gs_bot[0, 0])
ax_cost.bar(agg_cost.keys(), agg_cost.values(), color=scenario_colors.values(), alpha=0.9)
ax_cost.set_title("Aggregate\nPremium Payment")
ax_cost.set_ylabel("($B)")
for i, v in enumerate(agg_cost.values()):
    ax_cost.text(i, v, f"{v:.1f}", ha="center", fontsize=7)

# Aggregate Affordability
ax_aff = fig.add_subplot(gs_bot[0, 1])
ax_aff.bar(agg_afford.keys(), agg_afford.values(), color=scenario_colors.values(), alpha=0.9)
ax_aff.axhline(y=2, color='black', linestyle='--', linewidth=1)
ax_aff.set_title("Avg. Household\nAffordability")
ax_aff.set_ylabel("Affordability Ratio (%)")
for i, v in enumerate(agg_afford.values()):
    ax_aff.text(i, v, f"{v:.1f}", ha="center", fontsize=7)

# Chronic & Extreme Loss
gs_loss = GridSpecFromSubplotSpec(nrows=2, ncols=1, subplot_spec=gs_bot[0, 2], hspace=0.75, height_ratios=[1, 1])

# Chronic (Avg_Pos)
ax_chronic = fig.add_subplot(gs_loss[0, 0])
ax_chronic.bar(agg_chronic.keys(), [val*100 for val in agg_chronic.values()], color=scenario_colors.values(), alpha=0.9)
ax_chronic.set_title("Chronic Loss:\nAvg. Premium Retention")
ax_chronic.set_ylim([65,90])
ax_chronic.set_ylabel("(%)")
ax_chronic.set_xticklabels([])

# Extreme (Avg_Neg)
ax_extreme = fig.add_subplot(gs_loss[1, 0])
ax_extreme.bar(agg_extreme.keys(), agg_extreme.values(), color=scenario_colors.values(), alpha=0.9)
ax_extreme.axhline(0, color="black", lw=1)
ax_extreme.set_title("Extreme Loss:\nAvg. Recovery Time")
ax_extreme.set_ylim([-9.5,-6])
ax_extreme.set_ylabel("(YR)")

# Add subplot labels (a), b), c)... in reading order
for i, ax in enumerate(fig.axes):
    if i==8 or i==9:
         ax.text(
            -0.08, 1.5, f"{string.ascii_lowercase[i]})",
            transform=ax.transAxes,
            fontsize=11, fontweight='bold',
            va='top', ha='right'
        )

    else:
        ax.text(
            -0.08, 1.17, f"{string.ascii_lowercase[i]})",
            transform=ax.transAxes,
            fontsize=11, fontweight='bold',
            va='top', ha='right'
        )

plt.tight_layout(rect=[0, 0.02, 0.96, 1])
if save:
    plt.savefig("Plots/Fig4_Scenarios.pdf", dpi=500, bbox_inches="tight")
plt.show()